In [12]:
# Cell 1: Environment Initialization & Dependencies
import os
from google.colab import drive

!apt-get update -qq > /dev/null
!apt-get install -y -qq zstd > /dev/null
!pip install -q langchain langchain-community langchain-core chromadb \
    sentence-transformers pypdf langchain-text-splitters --no-cache-dir

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Capstone_RAG'
PDF_DIR     = os.path.join(BASE_DIR, 'sports_pdfs')
DB_DIR      = os.path.join(BASE_DIR, 'science_db_pro')
CSV_PATH    = os.path.join(BASE_DIR, 'East_Africa_Food_Dataset_FINAL.csv')

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DB_DIR,  exist_ok=True)

print(f"✅ Environment ready: {BASE_DIR}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Environment ready: /content/drive/MyDrive/Capstone_RAG


In [13]:
# Cell 2: RAG Vector Store Initialization
import warnings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

warnings.filterwarnings("ignore")

def initialize_vector_store(pdf_dir, db_dir):
    print("Initializing embedding model...")
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

    if os.path.exists(db_dir) and os.listdir(db_dir):
        print(f"✅ Loading existing vector index from: {db_dir}")
        return Chroma(persist_directory=db_dir, embedding_function=embeddings)

    print("Building new vector index from PDFs...")
    loader = PyPDFDirectoryLoader(pdf_dir)
    documents = loader.load()

    if not documents:
        print("⚠️ No PDF documents found.")
        return None

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_dir
    )
    print(f"✅ Indexed {len(chunks)} chunks from {len(documents)} pages.")
    return db

vector_store = initialize_vector_store(PDF_DIR, DB_DIR)

Initializing embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loading existing vector index from: /content/drive/MyDrive/Capstone_RAG/science_db_pro


In [14]:
# Cell 3: LLM Setup — Groq API (Llama 3, fast inference)
# Get free API key at: console.groq.com
# Install: pip install langchain-groq

!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq

GROQ_API_KEY = ""

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.7,
    max_tokens=1024
)

# Quick test
test = llm.invoke("Say: Groq is ready")
print("✅ Groq LLM ready — ultra fast inference active.")

✅ Groq LLM ready — ultra fast inference active.


In [15]:
# Cell 4: Load All Resources + Athlete Profile Schema
import pandas as pd
import re
import json
from dataclasses import dataclass
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Load Resources ──
print("Loading resources...")
embeddings   = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
# llm already loaded in Cell 3

try:
    food_df = pd.read_csv(CSV_PATH)
    print(f"✅ Food dataset loaded: {len(food_df)} items.")
except FileNotFoundError:
    print(f"⚠️ Dataset not found at {CSV_PATH}")

# ── Athlete Profile Schema ──
@dataclass
class AthleteProfile:
    """
    Structured athlete profile.
    Each field maps to a peer-reviewed nutritional adjustment:
      weight_kg     → base for all g/kg calculations (ACSM, IOC)
      age_group     → youth/veteran adjustments (IOC 2012)
      sex           → female iron & carb adjustments (ACSM 2009)
      sport         → activity classification (UEFA BJSM 2019)
      intensity     → carb multiplier base (ACSM)
      duration_mins → glycogen depletion window (IOC)
      goal          → carb periodization modifier (Burke et al. 2011)
    """
    weight_kg:     float
    age_group:     str
    sex:           str
    sport:         str
    intensity:     str
    duration_mins: float
    goal:          str

    def summary(self):
        return (
            f"{self.sex.title()} | {self.age_group} | {self.sport} | "
            f"{self.duration_mins:.0f} min | {self.intensity} intensity | "
            f"Goal: {self.goal} | {self.weight_kg}kg"
        )

print("✅ All resources ready.")

Loading resources...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Food dataset loaded: 203 items.
✅ All resources ready.


In [18]:
# Cell 5: Sports Nutrition AI Agent
# Scientific references embedded in calculation logic:
#   [1] ACSM Position Stand: Nutrition & Athletic Performance (2009)
#   [2] IOC Nutrition for Athletes — Maughan & Burke (2012)
#   [3] UEFA Expert Group Consensus Statement — Collins et al. (BJSM, 2019)
#   [4] FIFA F-MARC Nutrition for Football (2006)
#   [5] Burke et al. — Carbohydrate Periodization (2011)

class NutritionAgent:

    def __init__(self, vector_store, food_df, llm):
        self.retriever = vector_store.as_retriever(search_kwargs={"k": 5})
        self.llm       = llm
        self.food_df   = food_df
        self._normalize_dataset()

    def _normalize_dataset(self):
        def extract(text, nutrient):
            m = re.search(rf'([0-9.]+)g\s+{nutrient}', str(text), re.IGNORECASE)
            return float(m.group(1)) if m else 0.0
        self.food_df['Carbs_Parsed']   = self.food_df['Macros per 100g (C / P / F)'].apply(lambda x: extract(x, 'Carbs'))
        self.food_df['Protein_Parsed'] = self.food_df['Macros per 100g (C / P / F)'].apply(lambda x: extract(x, 'Pro'))
        print(f"✅ {len(self.food_df)} foods normalized.")

    def _gatekeeper_check(self, text: str):
        prompt = f"""
        Analyze: "{text}"
        Is this a physical sport, exercise, or athletic activity requiring sports nutrition?
        Output ONLY valid JSON, no explanation, no markdown:
        {{"is_sport": true or false, "reason": "short friendly explanation if false"}}
        """
        result = self.llm.invoke(prompt)
        raw = result.content if hasattr(result, 'content') else str(result)
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            d = json.loads(m.group(0))
            return d.get('is_sport', True), d.get('reason', '')
        except:
            return True, ""

    def _auto_detect(self, text: str) -> dict:
        prompt = f"""
        Analyze this athlete description and extract every field possible: "{text}"

        EXTRACTION RULES — be aggressive, extract everything:
        - If user said a sport name → that is "sport"
        - If user said hours or minutes → convert to "duration_mins"
        - If user said a number with kg or lbs → that is "weight_kg"
        - If user said an age or "years old" → derive "age_group"
        - If user said male/man/boy/guy → sex is "male"
        - If user said female/woman/girl → sex is "female"
        - If user said recovery/muscle/weight loss/performance → that is "goal"
        - intensity: basketball/soccer/football/rugby/sprinting = "heavy", jogging/cycling = "moderate", walking/yoga = "light". Always guess, never null.
        - age_group: under 18 = "youth", 18-40 = "adult", over 40 = "veteran"
        - goal default: "recovery" for any post-game or post-workout context

        Only add a field to "missing" if the user truly did not mention it at all.

        Output ONLY valid JSON, no markdown, no explanation:
        {{
            "weight_kg": 68.0,
            "age_group": "adult",
            "sex": "male",
            "sport": "basketball",
            "intensity": "heavy",
            "duration_mins": 180.0,
            "goal": "recovery",
            "missing": []
        }}
        """
        result = self.llm.invoke(prompt)
        raw = result.content if hasattr(result, 'content') else str(result)
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            return json.loads(m.group(0))
        except:
            return {
                "weight_kg": None, "age_group": None, "sex": None,
                "sport": "sport", "intensity": "moderate",
                "duration_mins": None, "goal": "recovery",
                "missing": ["weight_kg", "sex", "age_group", "duration_mins"]
            }

    def _ask_followup(self, missing: list, sport: str) -> dict:
        if not missing:
            return {}
        questions = {
            "weight_kg": (f"⚖️  What's your body weight?\n   (e.g. '70kg' or '154lbs')\n   This is the foundation of your carb and protein targets."),
            "duration_mins": (f"⏱️  How long was your {sport} session?\n   (e.g. '90 minutes' or '2 hours')"),
            "sex": (f"👤  Are you male or female?\n   This affects iron and carbohydrate recommendations."),
            "age_group": (f"🎂  How old are you?\n   Youth and veteran athletes have different nutritional needs."),
            "goal": (f"🎯  What's your main goal right now?\n   Options: recovery / muscle gain / weight loss / performance"),
        }
        answers = {}
        print("\n" + "─" * 55)
        print("💬 A few quick questions to personalize your plan:\n")
        for field in missing:
            if field not in questions:
                continue
            print(f"   {questions[field]}")
            answer = input("   >> ").strip()
            print()
            if field == "weight_kg":
                nums = re.findall(r'[\d.]+', answer)
                if nums:
                    w = float(nums[0])
                    answers["weight_kg"] = round(w / 2.2, 1) if 'lb' in answer.lower() else w
            elif field == "duration_mins":
                nums = re.findall(r'[\d.]+', answer)
                if nums:
                    val = float(nums[0])
                    answers["duration_mins"] = val * 60 if 'hour' in answer.lower() else val
            elif field == "sex":
                answers["sex"] = ("female" if any(w in answer.lower() for w in ["female", "woman", "girl", "f"]) else "male")
            elif field == "age_group":
                nums = re.findall(r'\d+', answer)
                if nums:
                    age = int(nums[0])
                    answers["age_group"] = ("youth" if age < 18 else "veteran" if age > 40 else "adult")
                else:
                    answers["age_group"] = "adult"
            elif field == "goal":
                a = answer.lower()
                if any(w in a for w in ["muscle", "gain", "bulk", "build"]):
                    answers["goal"] = "muscle_gain"
                elif any(w in a for w in ["loss", "lose", "cut", "slim", "weight"]):
                    answers["goal"] = "weight_loss"
                elif any(w in a for w in ["perform", "speed", "power", "compete"]):
                    answers["goal"] = "performance"
                else:
                    answers["goal"] = "recovery"
        print("─" * 55)
        return answers

    def _calculate_macros(self, p: AthleteProfile) -> dict:
        base     = {'light': 0.5, 'moderate': 0.8, 'heavy': 1.0}.get(p.intensity, 0.8)
        dur_adj  = 0.2 if p.duration_mins > 90 else (-0.2 if p.duration_mins < 45 else 0.0)
        age_adj  = {'youth': 0.1, 'adult': 0.0, 'veteran': -0.1}.get(p.age_group, 0.0)
        sex_adj  = -0.1 if p.sex.lower() == 'female' else 0.0
        goal_adj = {'muscle_gain': 0.2, 'performance': 0.1, 'recovery': 0.0, 'weight_loss': -0.2}.get(p.goal, 0.0)
        multiplier     = round(max(0.3, min(base + dur_adj + age_adj + sex_adj + goal_adj, 1.5)), 2)
        target_carbs   = round(p.weight_kg * multiplier, 1)
        prot_mult      = 1.6 if p.goal == 'muscle_gain' else 1.2
        if p.age_group == 'veteran':
            prot_mult += 0.1
        target_protein = round(p.weight_kg * prot_mult, 1)
        print(f"\n⚙️  Macro Calculation Breakdown:")
        print(f"   Base ({p.intensity}):         {base:+.1f} g/kg  [ACSM 2009]")
        print(f"   Duration ({p.duration_mins:.0f} min):  {dur_adj:+.1f} g/kg  [IOC 2012]")
        print(f"   Age ({p.age_group}):          {age_adj:+.1f} g/kg  [IOC 2012]")
        print(f"   Sex ({p.sex}):          {sex_adj:+.1f} g/kg  [ACSM 2009]")
        print(f"   Goal ({p.goal}):   {goal_adj:+.1f} g/kg  [Burke 2011]")
        print(f"   ─────────────────────────────────────────")
        print(f"   Final multiplier:       {multiplier} g/kg")
        print(f"   🍚 Carbohydrate target: {target_carbs}g")
        print(f"   🍗 Protein target:      {target_protein}g")
        return {
            'multiplier':     multiplier,
            'target_carbs':   target_carbs,
            'target_protein': target_protein,
            'prot_mult':      prot_mult,
            'weight_kg':      p.weight_kg,
            'breakdown': (f"Base {base} + duration {dur_adj:+} + age {age_adj:+} + sex {sex_adj:+} + goal {goal_adj:+} = {multiplier} g/kg")
        }

    def _get_meal(self, target_carbs: float, p: AthleteProfile) -> dict:
        def safe_sample(df):
            return df.sample(n=1) if not df.empty else pd.DataFrame()

        # ── Carbohydrate source — exclude junk/processed foods ──
        junk_exclude = ['Biscuit', 'Mandazi', 'Corn Flakes', 'Sweet Bread', 'Chapati']
        junk_pattern = '|'.join(junk_exclude)

        carb_pool = self.food_df[
            self.food_df['Primary Category'].str.contains(
                'Slow Carbohydrate|Fast Carbohydrate|Carbohydrate', case=False, na=False
            ) &
            (self.food_df['Carbs_Parsed'] > 0) &
            (~self.food_df['Food Item'].str.contains(junk_pattern, case=False, na=False)) &
            (~self.food_df['Valid Preparation'].str.contains('Deep Fried', case=False, na=False))
        ].sort_values('Carbs_Parsed', ascending=False).head(15)

        if carb_pool.empty:
            carb_pool = self.food_df[self.food_df['Carbs_Parsed'] > 0].sort_values('Carbs_Parsed', ascending=False).head(15)

        carb_choice = safe_sample(carb_pool)

        # ── Protein source — exclude seeds/nuts/powders as primary protein ──
        snack_exclude = ['Seeds', 'Nuts', 'Peanut Butter', 'Powder', 'Oil', 'Macadamia', 'Walnut', 'Almond']
        snack_pattern = '|'.join(snack_exclude)

        if p.sex.lower() == 'female':
            prot_pool = self.food_df[
                self.food_df['Primary Category'].str.contains(
                    'Lean Protein|Protein|Micronutrient', case=False, na=False
                ) &
                (self.food_df['Protein_Parsed'] > 0) &
                (~self.food_df['Food Item'].str.contains(snack_pattern, case=False, na=False))
            ].sort_values('Protein_Parsed', ascending=False).head(10)
            iron_note = "Iron-rich protein prioritized (ACSM female athlete guidelines)"
        else:
            prot_pool = self.food_df[
                self.food_df['Primary Category'].str.contains(
                    'Lean Protein|Protein', case=False, na=False
                ) &
                (self.food_df['Protein_Parsed'] > 0) &
                (~self.food_df['Food Item'].str.contains(snack_pattern, case=False, na=False))
            ].sort_values('Protein_Parsed', ascending=False).head(10)
            iron_note = ""

        if prot_pool.empty:
            prot_pool = self.food_df[self.food_df['Protein_Parsed'] > 0].sort_values('Protein_Parsed', ascending=False).head(10)

        prot_choice = safe_sample(prot_pool)

        # ── Hydration — strictly Hydration category ──
        liquid_pool = self.food_df[
            self.food_df['Primary Category'].str.contains('Hydration', case=False, na=False)
        ]
        if liquid_pool.empty:
            liquid_pool = self.food_df[
                self.food_df['Food Item'].str.contains(
                    'Water|Juice|Milk|Tea|Coconut|Drink', case=False, na=False
                )
            ]
        if liquid_pool.empty:
            liquid_pool = self.food_df.tail(5)

        liquid_choice = safe_sample(liquid_pool)

        # ── Build output strings ──
        carb_str = "Local starch base"
        if not carb_choice.empty:
            row   = carb_choice.iloc[0]
            grams = int(round((target_carbs / row['Carbs_Parsed']) * 100, -1))
            name  = row['Food Item'].replace('Raw ', '').replace('Dry ', '')
            carb_str = f"{name}: {grams}g"

        prot_str = (prot_choice.iloc[0]['Food Item'].replace('Raw ', '') if not prot_choice.empty else "Lean protein source")
        liquid_str = (liquid_choice.iloc[0]['Food Item'] if not liquid_choice.empty else "Water with a pinch of salt")

        return {'carb_str': carb_str, 'prot_str': prot_str, 'liquid_str': liquid_str, 'iron_note': iron_note}

    def _synthesize(self, p: AthleteProfile, macros: dict, meal: dict, docs: list) -> str:
        rag_context = ""
        print("\n📄 Scientific sources retrieved:")
        for i, doc in enumerate(docs):
            source = doc.metadata.get('source', 'unknown').split('/')[-1]
            snippet = doc.page_content[:150].strip().replace('\n', ' ')
            print(f"   [{i+1}] {source}")
            print(f"        {snippet}...")
            rag_context += f"\n[Source {i+1}: {source}]\n{doc.page_content[:300]}\n"

        template = f"""
You are a professional sports nutrition AI built for East African athletes.
Use this peer-reviewed scientific knowledge to inform your response:
{rag_context}

Athlete Profile:
- Sport: {p.sport}
- Age Group: {p.age_group}
- Sex: {p.sex}
- Duration: {p.duration_mins:.0f} minutes
- Intensity: {p.intensity}
- Goal: {p.goal}
- Weight: {p.weight_kg}kg

TASK: Write a warm, friendly, professional personalized recovery meal plan.

STRICT RULES:
1. First sentence must reference their SPECIFIC sport and duration directly.
   NEVER start with: "Great job!", "As an athlete", "I'm excited", "Here is a warm", "Certainly!", "Sure!"
   START like this: "Three hours of basketball significantly depletes muscle glycogen — here is exactly what you need."
2. Never use filler openers or hype phrases.
3. The Nutrition Tip must be about FOOD, TIMING, or NUTRIENTS only. Never mention exercise or stretching.

OUTPUT FORMAT — copy exactly:

[One direct sentence: sport + duration + physiological consequence]

**✅ What Your Body Needs Right Now**
* **Carbohydrates:** {macros['target_carbs']}g
  _{macros['breakdown']}_
* **Protein:** {macros['target_protein']}g
  _({macros['prot_mult']}g/kg × {macros['weight_kg']}kg — ACSM guideline for {p.goal})_
* **Hydration:** Minimum 500ml water + electrolyte replenishment

**🥗 Your Recovery Plate**
* **🍚 Carb Source:** {meal['carb_str']}
* **🍗 Protein Source:** {meal['prot_str']}
* **💧 Hydration:** {meal['liquid_str']}
{f"* **🩸 Iron Note:** {meal['iron_note']}" if meal['iron_note'] else ""}

**💡 Nutrition Tip**
[Specific tip about food timing or nutrients for {p.sport} {p.goal}. No exercise advice.]
"""
        result = self.llm.invoke(template)
        return result.content if hasattr(result, 'content') else str(result)

    def execute(self):
        print("\n" + "=" * 60)
        print("   🏋️  Sports Nutrition AI — East African Food Intelligence")
        print("=" * 60)
        print("\n👋 Hi! Tell me about your workout and I'll build a")
        print("   personalized recovery meal plan for you.")
        print("\n   Share as much as you can — sport, duration, weight,")
        print("   age, sex, and your goal. Don't worry if you forget")
        print("   something, I'll ask! 😊\n")

        workout_query = input(">> ")

        print("\n🔍 Analyzing your activity...")
        is_sport, reason = self._gatekeeper_check(workout_query)
        if not is_sport:
            print("\n" + "=" * 60)
            print(f"\n😊 {reason}")
            print("\n   Come back after your next training session!")
            print("=" * 60)
            return

        print("🧠 Reading your description...")
        detected = self._auto_detect(workout_query)
        missing  = detected.pop("missing", [])

        followup = self._ask_followup(missing, detected.get("sport", "workout"))
        detected.update(followup)

        defaults = {
            "weight_kg": 70.0, "age_group": "adult", "sex": "male",
            "sport": "general sport", "intensity": "moderate",
            "duration_mins": 60.0, "goal": "recovery"
        }
        for k, v in defaults.items():
            if not detected.get(k):
                detected[k] = v

        profile = AthleteProfile(
            weight_kg     = float(detected["weight_kg"]),
            age_group     = detected["age_group"],
            sex           = detected["sex"],
            sport         = detected["sport"],
            intensity     = detected["intensity"],
            duration_mins = float(detected["duration_mins"]),
            goal          = detected["goal"]
        )

        print(f"\n✅ Profile confirmed:")
        print(f"   {profile.summary()}\n")

        macros = self._calculate_macros(profile)

        print("\n📚 Retrieving from scientific knowledge base...")
        rag_query = (f"{profile.sport} {profile.intensity} nutrition {profile.goal} {profile.sex} {profile.age_group}")
        docs = self.retriever.invoke(rag_query)
        print(f"   Retrieved {len(docs)} relevant scientific chunks.")

        print("🥗 Selecting from East African food database...")
        meal = self._get_meal(macros['target_carbs'], profile)

        print("✨ Generating your personalized meal plan...\n")
        response = self._synthesize(profile, macros, meal, docs)

        print("\n" + "=" * 60)
        print(response)
        print("=" * 60)
        print("\n💪 Fuel well and recover strong!")

# ── RUN ──
agent = NutritionAgent(vector_store, food_df, llm)
agent.execute()

✅ 203 foods normalized.

   🏋️  Sports Nutrition AI — East African Food Intelligence

👋 Hi! Tell me about your workout and I'll build a
   personalized recovery meal plan for you.

   Share as much as you can — sport, duration, weight,
   age, sex, and your goal. Don't worry if you forget
   something, I'll ask! 😊

>> I played basketball for 2 hours with my freinds

🔍 Analyzing your activity...
🧠 Reading your description...

───────────────────────────────────────────────────────
💬 A few quick questions to personalize your plan:

   ⚖️  What's your body weight?
   (e.g. '70kg' or '154lbs')
   This is the foundation of your carb and protein targets.
   >> 68kg

   🎂  How old are you?
   Youth and veteran athletes have different nutritional needs.
   >> 22

   👤  Are you male or female?
   This affects iron and carbohydrate recommendations.
   >> male

───────────────────────────────────────────────────────

✅ Profile confirmed:
   Male | adult | basketball | 120 min | heavy intensity | 